In [8]:
%pip install msoffcrypto-tool
%pip install pywin32

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [9]:
%pip install python-calamine

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import msoffcrypto
import pandas as pd
import io
import openpyxl
import glob
import os


In [11]:
pd.set_option('display.max_rows', None)

In [18]:
password = '18651865'
cols = 'A:H,T,W:AC,Aj'
data = []

In [ ]:
import os 
import glob
import io
import msoffcrypto
import pandas as pd

path = r'X:\My Drive\การเงิน\ยอดขาย\[0-9][0-9][0-9][0-9]'

# 1. ค้นหาไฟล์ด้วย glob
search = os.path.join(path , "**" , 'ยอดขายรวมทุกสาขาBplus[0-9][0-9][0-9][0-9].xlsx')
files_year = glob.glob(search, recursive=True)

pattern_clean = os.path.join(path, "**", "ยอดขายรวมทุกสาขาBplus.xlsx")
files_clean = glob.glob(pattern_clean, recursive=True)

raw_excel_files = files_year + files_clean 

# --- [แก้ไขบัค 1] คัดกรองไฟล์ชั่วคราว (~$) ออกจากลิสต์ที่จะใช้งานจริง ---
excel_files = []
for file_path in raw_excel_files:
    if not os.path.basename(file_path).startswith('~$'):
        excel_files.append(file_path)

print(f"เจอไฟล์ Excel ทั้งหมดที่ใช้งานได้ {len(excel_files)} ไฟล์:")
for p in excel_files:
    print("- ", p)

# --- [แก้ไขบัค 4] ประกาศตัวแปรเก็บ DataFrame ไว้ที่นอกลูปหลัก ---
data = []

# 2. เริ่มลูปแกะไฟล์รายไฟล์
for i in excel_files:
    print(f"กำลังประมวลผลไฟล์: {os.path.basename(i)}")
    
    # --- [แก้ไขบัค 2] สร้าง BytesIO ใหม่ทุกครั้งที่เริ่มไฟล์ใหม่ ---
    temp_file = io.BytesIO()    
    
    with open(i, 'rb') as f:
        office_file = msoffcrypto.OfficeFile(f)
        
        # ใส่รหัสผ่านเพื่อปลดล็อก
        office_file.load_key(password=password)
        
        # บันทึกไฟล์ที่ปลดล็อกแล้วลงในหน่วยความจำ
        office_file.decrypt(temp_file)

        # ชี้ตำแหน่ง Pointer ของ BytesIO กลับไปที่จุดเริ่มต้นก่อนให้อ่านข้อมูล
        temp_file.seek(0)
        
        da = pd.ExcelFile(temp_file)
        
        # ปฏิเสธตัวที่ไม่เอาตรงๆ (เอาเฉพาะชีทประจำเดือน)
        month_names = [x for x in da.sheet_names if not x.startswith('รวม') and '.' in x]
        
        # ลูปอ่านแต่ละชีทในไฟล์นั้นๆ
        for m in month_names:
            df = pd.read_excel(temp_file, sheet_name=m, usecols=cols, header=4, engine='calamine')
            df = df.iloc[0:35, :]
            
            # แปลงข้อมูลเป็นวันที่
            df['DATE'] = pd.to_datetime(df['Unnamed: 0'],  format='%Y/%m/%d', errors='coerce')
            
            # ลบค่าว่างในคอลัมน์ DATE
            df.dropna(subset=['DATE'], inplace=True)
            
            # กรองวันที่ที่มากกว่า 2020
            df = df[df['DATE'].dt.year > 2020]

            df['DATE'] = df['DATE'].dt.date
            df.drop(columns=['Unnamed: 0'], inplace=True)
            df.rename(columns={'Unnamed: 19': 'WH'}, inplace=True)

            # เก็บ DataFrame ของชีทนี้เข้าลิสต์หลัก
            data.append(df)

# --- [แก้ไขบัค 3] ย้ายการรวมร่างไฟล์ (Concat) และคลีนข้อมูลมาไว้ "นอกลูปหลัก" ---
if data:  # เช็คก่อนว่ามีข้อมูลถูกเก็บมาจริงไหม
    print("กำลังรวมข้อมูลจากทุกไฟล์และทุกชีท...")
    full = pd.concat(data, ignore_index=True)
    
    # ลบข้อมูลซ้ำ
    full.drop_duplicates(inplace=True)

    # จัดการย้ายคอลัมน์ DATE ไปไว้หน้าสุด
    if 'DATE' in full.columns:
        full.insert(0, 'DATE1', full['DATE'])
        full.drop(columns=['DATE','ค้าปลีก+ส่ง'], inplace=True)
        full.rename(columns={'DATE1': 'DATE','รวมยอดบิล.1':'ยอดบิล WH'}, inplace=True)
        
    print(f"รวมข้อมูลเสร็จสิ้น! ได้ข้อมูลทั้งหมด {len(full)} แถว")
else:
    print("ไม่พบข้อมูลที่จะนำมารวมกัน")
    full = pd.DataFrame() # ส่ง Dataframe เปล่ากลับไป

เจอไฟล์ Excel ทั้งหมดที่ใช้งานได้ 4 ไฟล์:
-  X:\My Drive\การเงิน\ยอดขาย\2569\ยอดขายรวมทุกสาขาBplus2569.xlsx
-  X:\My Drive\การเงิน\ยอดขาย\2567\ยอดขายรวมทุกสาขาBplus2567.xlsx
-  X:\My Drive\การเงิน\ยอดขาย\2568\ยอดขายรวมทุกสาขาBplus2568.xlsx
-  X:\My Drive\การเงิน\ยอดขาย\2570\ยอดขายรวมทุกสาขาBplus.xlsx
กำลังประมวลผลไฟล์: ยอดขายรวมทุกสาขาBplus2569.xlsx
กำลังประมวลผลไฟล์: ยอดขายรวมทุกสาขาBplus2567.xlsx
กำลังประมวลผลไฟล์: ยอดขายรวมทุกสาขาBplus2568.xlsx
กำลังประมวลผลไฟล์: ยอดขายรวมทุกสาขาBplus.xlsx
กำลังรวมข้อมูลจากทุกไฟล์และทุกชีท...
รวมข้อมูลเสร็จสิ้น! ได้ข้อมูลทั้งหมด 1096 แถว


In [ ]:
full = full.sort_values(by=['DATE'],ascending=True).reset_index(drop=True)
full = full.drop(columns=['index'], errors='ignore')

In [38]:
full.to_excel(r'C:\Users\KS\Desktop\Mydata.xlsx',index=False,engine='openpyxl')